# XAI & Destilación — Deep Dive

Este notebook asume que ya tienes un checkpoint entrenado (`Q_theta.ckpt`) y el dataset MDP (`D_offline.npz`).  
Usa **BPI2020-Travel** (~7k casos, el más pequeño) para experimentar rápido con XAI.

### Contenido
1. Setup rápido (pipeline desde cero si no hay checkpoint)
2. Cargar la red Q entrenada
3. Integrated Gradients — Risk (φ^V)
4. Integrated Gradients — ΔQ (φ^ΔQ)
5. Explicaciones completas con `explain_policy`
6. Tests de fidelidad (Q-drop, action-flip, rank-consistency)
7. Policy summary (clustering de estados)
8. Destilación VIPER → árbol de decisión
9. Exportar reglas SQL y usar el árbol sin GPU

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import torch

import xppm

print("xppm version:", xppm.__version__)
print("CUDA disponible:", torch.cuda.is_available())

BASE = Path("..")
DATA = BASE / "data"
ART = BASE / "artifacts"
CFG = BASE / "configs" / "config.yaml"
DS = "bpi2020-travel"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 0. Setup rápido (ejecutar solo si no tienes checkpoint)

Si ya tienes `Q_theta.ckpt`, salta esta celda.

In [ ]:
from xppm import (
    Config,
    EventLogSchema,
    LifecycleConfig,
    OutcomeConfig,
    TDQNConfig,
    build_mdp_dataset,
    encode_prefixes,
    preprocess_event_log,
    train_tdqn,
    validate_and_split_dataset,
)

cfg = Config.for_dataset(CFG, DS)

raw_path = DATA / "raw" / "bpi2020-travel-permit-data.xes.gz"
clean_path = DATA / DS / "interim" / "clean.parquet"
prefixes_path = DATA / DS / "interim" / "prefixes.npz"
vocab_path = DATA / DS / "interim" / "vocab_activity.json"
mdp_path = DATA / DS / "processed" / "D_offline.npz"
splits_path = DATA / DS / "processed" / "splits.json"
ckpt_dir = ART / "models" / "tdqn" / "bpi2020travel_demo"
ckpt_path = ckpt_dir / "Q_theta.ckpt"

if not ckpt_path.exists():
    print("Checkpoint no encontrado — ejecutando pipeline...")
    for d in [clean_path.parent, mdp_path.parent, ckpt_dir]:
        d.mkdir(parents=True, exist_ok=True)

    schema = EventLogSchema(
        case_id="case:concept:name",
        activity="concept:name",
        timestamp="time:timestamp",
        select_cols=["case_id", "activity", "timestamp"],
        lifecycle=LifecycleConfig(enabled=False),  # BPI2020-travel no tiene lifecycle
        outcome=OutcomeConfig(
            mode="from_activity",
            positive_activities=["Payment Handled"],
        ),
    )

    preprocess_event_log(raw_path, clean_path, schema=schema)
    cfg.raw["encoding"]["output"]["vocab_activity_path"] = str(vocab_path)
    encode_prefixes(clean_path, prefixes_path, config=cfg.raw)
    build_mdp_dataset(prefixes_path, clean_path, vocab_path, mdp_path, config=cfg.raw)
    validate_and_split_dataset(mdp_path, splits_path, config=cfg.raw)

    tdqn_cfg = TDQNConfig(
        npz_path=str(mdp_path),
        splits_path=str(splits_path),
        vocab_path=str(vocab_path),
        max_len=50,
        d_model=128,
        n_heads=4,
        n_layers=3,
        n_actions=2,
        batch_size=128,
        learning_rate=3e-4,
        gamma=0.99,
        max_steps=10_000,  # demo; producción: 50_000
        eval_every=2_000,
        save_every=10_000,
        double_dqn=True,
        target_update_every=1000,
        grad_clip_norm=10.0,
        device=str(DEVICE),
        seed=42,
    )
    train_tdqn(tdqn_cfg, checkpoint_dir=ckpt_dir)
    print("Pipeline completado. Checkpoint:", ckpt_path)
else:
    print(f"Checkpoint encontrado: {ckpt_path}")
    cfg = Config.for_dataset(CFG, DS)
    cfg.raw["encoding"]["output"]["vocab_activity_path"] = str(vocab_path)

## 1. Cargar la red Q entrenada

`AgentFactory` infiere la arquitectura desde el checkpoint y el dataset — sin hardcodear nada.

In [ ]:
from xppm import AgentFactory
from xppm.rl.train_tdqn import load_dataset_with_splits
from xppm.utils.io import load_json

q_net = AgentFactory.load(
    ckpt_path=ckpt_path,
    npz_path=mdp_path,
    vocab_path=vocab_path,
    config=cfg.raw,
    device=DEVICE,
)
print(f"Red: {q_net.__class__.__name__}")
print(f"  vocab_size:  {q_net.vocab_size}")
print(f"  d_model:     {q_net.d_model}")
print(f"  n_actions:   {q_net.n_actions}")
print(f"  parámetros:  {sum(p.numel() for p in q_net.parameters()):,}")

# Datos de test
test_data = load_dataset_with_splits(mdp_path, splits_path, split="test")
s_test = test_data["s"]
sm_test = test_data["s_mask"]
va_test = test_data["valid_actions"]
cp_test = test_data["case_ptr"]
tp_test = test_data.get("t_ptr", np.arange(len(s_test)))

vocab = load_json(vocab_path)
id2token = vocab.get("id2token", [])
action_names = cfg.raw["mdp"]["actions"]["id2name"]

print(f"\nTest set: {len(s_test):,} transiciones de {len(np.unique(cp_test)):,} casos")

## 2. Integrated Gradients — Risk (φ^V)

`compute_attributions` calcula IG directamente sobre un batch de estados.  
Útil para exploración manual o análisis puntual fuera del pipeline completo.

In [ ]:
from xppm import compute_attributions

# Muestra aleatoria para demo
rng = np.random.default_rng(42)
idx = rng.choice(len(s_test), size=min(30, len(s_test)), replace=False)
s_samp, sm_samp, va_samp = s_test[idx], sm_test[idx], va_test[idx]

xai_cfg = cfg.raw.get("xai", {})
xai_cfg.setdefault("methods", {})["risk"] = {
    "baseline": "pad",
    "n_steps_ig": 32,  # producción: 128
}

risk_attr = compute_attributions(
    q_net=q_net,
    states=s_samp,
    state_masks=sm_samp,
    valid_actions=va_samp,
    config=xai_cfg,
    device=DEVICE,
    target="V",
)

ig = risk_attr.get("ig_completeness", {})
print(
    f"IG completitud — abs_err: {ig.get('mean_abs_err', 'N/A'):.4f}  "
    f"rel_err (mediana): {ig.get('median_rel_err', 'N/A'):.4f}"
)
print("  → ideal: abs_err < 0.1")

# Caso de mayor riesgo
i = np.argmax(risk_attr["v_s"])
tok_imp = risk_attr["token_importance"][i]
real_pos = np.where(sm_samp[i].astype(bool))[0]
top_pos = real_pos[np.argsort(tok_imp[real_pos])[::-1][:6]]

print(
    f"\nCaso más arriesgado — V(s)={risk_attr['v_s'][i]:.4f}, "
    f"acción: {action_names[int(risk_attr['a_star'][i])]}"
)
for pos in top_pos:
    tid = int(s_samp[i, pos])
    name = str(id2token[tid]) if 0 <= tid < len(id2token) else f"id_{tid}"
    bar = "█" * max(1, int(tok_imp[pos] / (tok_imp[top_pos[0]] + 1e-12) * 15))
    print(f"  [{pos:2d}] {name:<35s} {bar} {tok_imp[pos]:.4f}")

## 3. Integrated Gradients — ΔQ (φ^ΔQ)

Explica **por qué el agente prefiere intervenir** vs. no hacer nada.

In [ ]:
noop_id = action_names.index("do_nothing") if "do_nothing" in action_names else 0

dq_attr = compute_attributions(
    q_net=q_net,
    states=s_samp,
    state_masks=sm_samp,
    valid_actions=va_samp,
    config=xai_cfg,
    device=DEVICE,
    target="deltaQ",
    contrast_action_id=noop_id,
)

delta_q = dq_attr["delta_q"]
print(f"ΔQ medio (positivo = TDQN prefiere intervenir): {delta_q.mean():.4f}")
print(f"Casos donde se recomienda intervenir: {(delta_q > 0).sum()} / {len(delta_q)}")

# Caso donde más conviene intervenir
j = np.argmax(delta_q)
tok_imp_dq = dq_attr["token_importance"][j]
real_j = np.where(sm_samp[j].astype(bool))[0]
top_j = real_j[np.argsort(tok_imp_dq[real_j])[::-1][:6]]

print(f"\nCaso con ΔQ={delta_q[j]:.4f} (mayor ventaja de intervenir):")
print("  Actividades que impulsan la intervención:")
for pos in top_j:
    tid = int(s_samp[j, pos])
    name = str(id2token[tid]) if 0 <= tid < len(id2token) else f"id_{tid}"
    bar = "█" * max(1, int(tok_imp_dq[pos] / (tok_imp_dq[top_j[0]] + 1e-12) * 15))
    print(f"  [{pos:2d}] {name:<35s} {bar} {tok_imp_dq[pos]:.4f}")

## 4. Explicaciones completas con `explain_policy`

`explain_policy` orquesta todo el paso XAI: selección de casos, IG risk + deltaQ,  
clustering de política, y guarda todos los artefactos en disco.

In [ ]:
from xppm import explain_policy

cfg.raw["xai"]["checkpoint_path"] = str(ckpt_path)
cfg.raw["xai"]["n_cases"] = 50
cfg.raw["xai"]["out_dir"] = f"xai/{DS}"
cfg.raw["xai"]["methods"]["risk"]["n_steps_ig"] = 64  # producción: 128

xai_paths = explain_policy(cfg.raw, config_hash=cfg.config_hash)

print("Artefactos XAI generados:")
for name, path in xai_paths.items():
    size_kb = Path(path).stat().st_size / 1024
    print(f"  {name:<20s} → {path.name}  ({size_kb:.0f} KB)")

In [ ]:
# Resumen global de tokens más importantes (frecuencia en top-k across all cases)
risk_data = json.loads(xai_paths["risk"].read_text())

print(f"Explicaciones para {len(risk_data['items'])} transiciones.")
print()
print("Top actividades globales (por frecuencia en top-k):")
for entry in risk_data["metadata"].get("top_tokens_risk", [])[:8]:
    name = entry.get("token_name", f"token_{entry['token_id']}")
    print(
        f"  {name:<40s}  f={entry['frequency']:3d}  "
        f"imp_mediana={entry['median_importance']:.4f}"
    )

## 5. Tests de fidelidad

Verifica que las explicaciones IG realmente capturan lo que importa para la política.

| Test | ¿Qué mide? |
|---|---|
| **Q-drop** | ¿Baja V cuando enmascaramos los tokens más importantes según IG? |
| **Action-flip** | ¿Cambia la acción recomendada al perturbar los top-k tokens? |
| **Rank-consistency** | Correlación Spearman entre ranking IG y ranking de impacto en Q |

In [ ]:
import pandas as pd

from xppm.xai.fidelity_tests import run_fidelity_tests

# Activar todos los tests y reducir n_random para demo
cfg.raw["fidelity"]["enabled"] = True
cfg.raw["fidelity"]["n_random"] = 5  # producción: 20
cfg.raw["fidelity"]["n_items"] = 50  # limitar a 50 items para demo
cfg.raw["fidelity"]["out_csv"] = f"artifacts/fidelity/{DS}/fidelity.csv"

# run_fidelity_tests lee los artefactos XAI y guarda resultados en CSV
run_fidelity_tests(cfg.raw)

# Leer y mostrar resultados
fidelity_csv = Path(cfg.raw["fidelity"]["out_csv"])
if fidelity_csv.exists():
    fid_df = pd.read_csv(fidelity_csv)
    print("Resultados de fidelidad:")
    print(fid_df.to_string(index=False))
else:
    print(f"CSV no encontrado en {fidelity_csv}")

## 6. Policy Summary — Clustering de estados

Agrupa los estados del test set en clusters para identificar **situaciones típicas** del proceso.

In [ ]:
from xppm.xai.policy_summary import summarize_policy

summary = summarize_policy(
    q_net=q_net,
    states=s_test,
    state_masks=sm_test,
    valid_actions=va_test,
    case_ptrs=cp_test,
    t_ptrs=tp_test,
    action_names=action_names,
    config=cfg.raw.get("xai", {}),
    device=DEVICE,
)

print(f"Clusters k={summary['n_clusters']}\n")
print(f"{'Cluster':>8} {'Tamaño':>8} {'Acción dominante':<26} {'V(s) medio':>12}")
print("-" * 60)
for cl in summary["clusters"]:
    print(
        f"  {cl['cluster_id']:>4}   {cl['size']:>8,}   "
        f"{cl['dominant_action']:<26} {cl['mean_v']:>10.4f}"
    )

## 7. Destilación VIPER → Árbol de decisión

`distill_policy` entrena un `DecisionTreeClassifier` sobre pares `(features_tabulares, acción_TDQN)`.  
Retorna las **rutas a los artefactos** generados (no el árbol en sí).

In [ ]:
from xppm import distill_policy

cfg.raw["distill"]["teacher_checkpoint"] = str(ckpt_path)
cfg.raw["distill"]["sample"]["n_states"] = 500  # demo; producción: 2000
cfg.raw["distill"]["surrogate"]["max_depth"] = 5
cfg.raw["distill"]["surrogate"]["min_samples_leaf"] = 10

distill_paths = distill_policy(cfg.raw)
# distill_paths = {"tree_pkl": Path, "fidelity_metrics": Path, "selection": Path}

# Leer métricas de fidelidad del árbol
fidelity_metrics = json.loads(distill_paths["fidelity_metrics"].read_text())
print("Fidelidad árbol vs TDQN:")
print(f"  train:  {fidelity_metrics.get('fidelity_train', 'N/A'):.3f}")
print(f"  test:   {fidelity_metrics.get('fidelity_test', 'N/A'):.3f}")
print("  → ideal > 0.85")

print("\nArtefactos generados:")
for key, path in distill_paths.items():
    print(f"  {key:<20s} → {path}")

In [ ]:
# Cargar el árbol desde tree.pkl para inspección
with open(distill_paths["tree_pkl"], "rb") as f:
    tree_bundle = pickle.load(f)

tree = tree_bundle["model"]
feature_names = tree_bundle["feature_names"]
action_names_ = tree_bundle["action_names"]

print(f"Árbol: profundidad={tree.get_depth()}, hojas={tree.get_n_leaves()}")
print(f"Features: {len(feature_names)}")
print()

# Feature importance
importances = tree.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
print("Top features usadas por el árbol:")
for i in sorted_idx[:10]:
    if importances[i] > 0:
        fname = feature_names[i] if i < len(feature_names) else f"feature_{i}"
        bar = "█" * int(importances[i] / (importances[sorted_idx[0]] + 1e-12) * 20)
        print(f"  {fname:<40s} {bar:20s} {importances[i]:.4f}")

## 8. Exportar reglas a SQL

Las reglas del árbol se exportan como queries SQL para auditoría o integración con sistemas existentes.

In [ ]:
from xppm.distill.export_rules import export_rules

rules_dir = distill_paths["tree_pkl"].parent
rule_paths = export_rules(
    tree_pkl_path=distill_paths["tree_pkl"],
    output_dir=rules_dir,
    config=cfg.raw,
)
# rule_paths = {"text": Path, "sql": Path, "metadata": Path}

print("Archivos exportados:")
for name, path in rule_paths.items():
    print(f"  {name}: {path}")

# Primeras líneas del SQL
print("\nSQL generado (primeras 25 líneas):")
sql_text = rule_paths["sql"].read_text()
for i, line in enumerate(sql_text.splitlines()[:25], 1):
    print(f"  {i:2d}: {line}")

## 9. Usar el árbol en producción (sin GPU)

El árbol destilado se puede cargar con scikit-learn y predecir en cualquier entorno,  
sin necesidad de PyTorch ni GPU.

In [ ]:
# El tree.pkl contiene todo lo necesario para predicción offline
n_features = tree.n_features_in_

# En producción recibirías estas features desde el event log en tiempo real
# Por ejemplo: [prefix_len, count_activity_A, count_activity_B, elapsed_time, ...]
example_state = np.zeros((1, n_features), dtype=np.float32)

pred_action_id = int(tree.predict(example_state)[0])
proba = tree.predict_proba(example_state)[0]

print("Estado de ejemplo (todos ceros):")
print(f"  Acción predicha: {action_names_[pred_action_id]} (id={pred_action_id})")
print()
print("  Probabilidades por acción:")
for name, p in zip(action_names_, proba):
    bar = "█" * int(p * 20)
    print(f"    {name:<30s} {bar:20s} {p:.3f}")

In [ ]:
# Reglas del árbol en texto legible
from sklearn.tree import export_text

rules_text = export_text(tree, feature_names=feature_names, max_depth=3)
print("Árbol destilado (primeras 3 capas):")
print(rules_text)